# 4 - QB - HighFrequencyImputer pas à pas

Ce notebook déroule, **phase par phase**, l'algorithme des méthodes `_fit` et `_transform`
de `HighFrequencyImputer` (`tsforecast/frequency/high_frequency_imputer.py`), en appelant
directement les mêmes méthodes privées que la classe utilise en interne (`_build_stage_frame`,
`_prepare_training_data`, `_apply_frequency_scaling`, `_predict_stage_values`,
`_apply_period_totals`, `_mark_imputed_cells`, ...).

**Pourquoi cette approche plutôt qu'une réécriture de l'algorithme ?** Parce qu'elle garantit
qu'on observe exactement ce que fait le code réel, sans risque de divergence entre ce que le
notebook montre et ce que la classe fait vraiment. La dernière section de chaque partie
(panel puis séries temporelles) le vérifie explicitement : le résultat obtenu pas à pas est
comparé, cellule par cellule, à un simple `imputer.fit_transform(X)` — l'écart maximal observé
doit être `0.0`.

À chaque étape, on affiche systématiquement :
- la **forme** (shape) des objets manipulés,
- le **nombre de NaN par colonne**,
- un **head/tail** pertinent,
- et les variables scalaires clés (fréquences, facteurs d'échelle, masques, etc.).

## Plan

1. Jeux de données (repris de `3 - QB - Panel a frequences mixtes heterogene.ipynb`)
2. Fonctions utilitaires d'affichage
3. **Partie panel** (le cas complexe : entités à couverture hétérogène, une variable dont la
   fréquence de publication diffère selon l'entité) — `_fit` détaillé phase par phase (PHASE
   0 à 6, exactement les phases commentées dans le code source), puis `_transform`
4. Vérification croisée panel vs `fit_transform()`
5. **Partie séries temporelles** (cas plus simple, une seule entité) — même cheminement
6. Vérification croisée séries temporelles vs `fit_transform()`

## 1 - Imports

In [ ]:
"""Refactored version of hfi_walkthrough.py: same validated logic, wrapped into
per-phase functions so it can be reused for both the panel and the time-series
walkthrough without duplicating ~500 lines. Run with `uv run python`.
"""
import warnings
from collections import OrderedDict
from dataclasses import replace

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.base import clone

from tsforecast.frequency import HighFrequencyImputer
from tsforecast.frequency.imputation_plan import ImputationStep, INTERPOLATE_FALLBACK, to_entity_tuple
from tsforecast.frequency.provenance import ImputationProvenanceTracker, ProvenanceType
from tsforecast.frequency.imputation_window import ImputationWindowCalculator
from tsforecast.panel.utils import (
    is_panel_data, get_unique_panel_entities, normalize_entity_key, split_variable_key,
    extract_column_names, group_keys_by_entity_and_variable, get_entity_mask,
)
from tsforecast.utils.frequency.utils import (
    normalize_frequency, is_higher_frequency, get_frequency_order,
    detect_frequency, detect_index_frequency,
)

try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
warnings.filterwarnings('ignore')

## 2 - Jeux de données

Les deux générateurs ci-dessous sont repris tels quels de
`notebooks/3 - QB - Panel a frequences mixtes heterogene.ipynb` (sections 2.1 et 2.2) : ils
donnent des jeux de données dont la complexité (fréquences mixtes, délais de publication,
historiques tronqués, et pour le panel : couverture temporelle hétérogène par entité et
fréquence de publication différente selon l'entité pour `depenses_publiques_pib`) reflète
fidèlement les cas rencontrés en pratique.

In [ ]:
import numpy as np
import pandas as pd


def create_timeseries_dataset(
    start_date: str = '2018-01-01',
    end_date: str = '2024-07-01',
    annual_start_date: str = '2015-01-01',
    seed: int = 42,
) -> pd.DataFrame:
    """Create a realistic macroeconomic time series dataset with mixed frequencies."""
    np.random.seed(seed)

    dates = pd.date_range(start=start_date, end=end_date, freq='MS')
    n_periods = len(dates)

    df = pd.DataFrame(index=dates)
    df.index.name = 'date'

    trend = np.linspace(100, 115, n_periods)
    seasonal = 3 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
    noise = np.random.normal(0, 1.5, n_periods)
    df['production_industrielle'] = trend + seasonal + noise

    inflation_trend = np.linspace(1.2, 2.8, n_periods)
    inflation_noise = np.random.normal(0, 0.3, n_periods)
    df['inflation_ipc'] = np.clip(inflation_trend + inflation_noise, 0.5, 5.0)

    chomage_trend = np.concatenate([
        np.linspace(8.5, 7.0, n_periods // 3),
        np.linspace(7.0, 9.5, n_periods // 3),
        np.linspace(9.5, 7.5, n_periods - 2 * (n_periods // 3)),
    ])
    chomage_noise = np.random.normal(0, 0.2, n_periods)
    df['taux_chomage'] = np.clip(chomage_trend + chomage_noise, 4.0, 15.0)

    pib_base = 2500
    pib_growth_quarterly = 0.5
    df['pib_trimestriel'] = np.nan

    quarter_start_months = [1, 4, 7, 10]
    quarter_idx = 0
    for i, date in enumerate(dates):
        if date.month in quarter_start_months:
            growth = pib_growth_quarterly + np.random.normal(0, 0.3)
            df.loc[date, 'pib_trimestriel'] = pib_base * (1 + growth / 100) ** quarter_idx
            quarter_idx += 1

    annual_dates = pd.date_range(start=annual_start_date, end=end_date, freq='YS')
    df = df.reindex(df.index.union(annual_dates))
    df.index.name = 'date'

    df['balance_commerciale_annuelle'] = np.nan
    for date in annual_dates:
        year_factor = (date.year - 2018)
        base_balance = -25 + year_factor * 3 + np.random.normal(0, 5)
        df.loc[date, 'balance_commerciale_annuelle'] = base_balance

    df.loc[df.index[-1], 'inflation_ipc'] = np.nan
    df.loc[df.index[-1], 'taux_chomage'] = np.nan

    pib_available = df[df['pib_trimestriel'].notna()].index
    if len(pib_available) > 0:
        df.loc[pib_available[-1], 'pib_trimestriel'] = np.nan

    bc_available = df[df['balance_commerciale_annuelle'].notna()].index
    if len(bc_available) > 0:
        df.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan

    mask_before_2019 = df.index < '2019-01-01'
    df.loc[mask_before_2019, 'production_industrielle'] = np.nan

    return df

In [ ]:
def create_panel_dataset(seed: int = 42) -> pd.DataFrame:
    """Create a realistic macroeconomic panel dataset with mixed frequencies."""
    np.random.seed(seed)
    countries = {
        'France': {
            'climat_affaires_observe': True,
            'pib_base': 2800, 'inflation_base': 1.5, 'chomage_base': 8.0, 'depenses_base': 55.0,
            'start_date': '2018-01-01', 'end_date': '2024-07-01',
            'prod_ind_start': '2018-06-01', 'depenses_frequency': 'annuelle',
            'annual_start_date': '2015-01-01',
        },
        'Allemagne': {
            'climat_affaires_observe': True,
            'pib_base': 3500, 'inflation_base': 1.2, 'chomage_base': 5.5, 'depenses_base': 45.0,
            'start_date': '2018-07-01', 'end_date': '2024-04-01',
            'prod_ind_start': '2019-01-01', 'depenses_frequency': 'trimestrielle',
            'annual_start_date': '2016-01-01',
        },
        'Italie': {
            'climat_affaires_observe': False,
            'pib_base': 2200, 'inflation_base': 1.8, 'chomage_base': 10.5, 'depenses_base': 50.0,
            'start_date': '2019-01-01', 'end_date': '2024-07-01',
            'prod_ind_start': '2019-06-01', 'depenses_frequency': 'annuelle',
            'annual_start_date': '2016-01-01',
        },
    }

    all_data = []
    for country, params in countries.items():
        np.random.seed(seed + hash(country) % 1000)

        dates = pd.date_range(start=params['start_date'], end=params['end_date'], freq='MS')
        n_periods = len(dates)

        df_country = pd.DataFrame(index=dates)
        df_country['country'] = country

        trend = np.linspace(100, 112 + np.random.uniform(-3, 3), n_periods)
        seasonal = 2.5 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
        noise = np.random.normal(0, 1.2, n_periods)
        df_country['production_industrielle'] = trend + seasonal + noise
        prod_start = pd.Timestamp(params['prod_ind_start'])
        df_country.loc[df_country.index < prod_start, 'production_industrielle'] = np.nan

        infl_trend = np.linspace(
            params['inflation_base'], params['inflation_base'] + np.random.uniform(0.5, 2.0), n_periods
        )
        infl_noise = np.random.normal(0, 0.25, n_periods)
        df_country['inflation_ipc'] = np.clip(infl_trend + infl_noise, 0.3, 6.0)

        chomage_base = params['chomage_base']
        chomage_evolution = np.concatenate([
            np.linspace(chomage_base, chomage_base - 1, n_periods // 3),
            np.linspace(chomage_base - 1, chomage_base + 2, n_periods // 3),
            np.linspace(chomage_base + 2, chomage_base + 0.5, n_periods - 2 * (n_periods // 3)),
        ])
        chomage_noise = np.random.normal(0, 0.15, n_periods)
        df_country['taux_chomage'] = np.clip(chomage_evolution + chomage_noise, 2.5, 15.0)

        df_country['pib_trimestriel'] = np.nan
        quarter_end_months = [1, 4, 7, 10]
        quarter_idx = 0
        for date in dates:
            if date.month in quarter_end_months:
                growth = 0.4 + np.random.normal(0, 0.35)
                df_country.loc[date, 'pib_trimestriel'] = params['pib_base'] * (1 + growth / 100) ** quarter_idx
                quarter_idx += 1

        df_country['depenses_publiques_pib'] = np.nan
        publication_months = [1] if params['depenses_frequency'] == 'annuelle' else [1, 4, 7, 10]
        depenses_idx = 0
        for date in dates:
            if date.month in publication_months:
                value = params['depenses_base'] + 0.1 * depenses_idx + np.random.normal(0, 1.0)
                df_country.loc[date, 'depenses_publiques_pib'] = value
                depenses_idx += 1

        annual_dates = pd.date_range(start=params['annual_start_date'], end=params['end_date'], freq='YS')
        df_country = df_country.reindex(df_country.index.union(annual_dates))
        df_country['country'] = country

        df_country['balance_commerciale_annuelle'] = np.nan
        for date in annual_dates:
            year_factor = (date.year - 2018)
            base = -20 + np.random.uniform(-10, 10) + year_factor * 2
            df_country.loc[date, 'balance_commerciale_annuelle'] = base

        # Climat des affaires : enquête de conjoncture mensuelle publiée pour la France
        # et l'Allemagne, jamais observée pour l'Italie (colonne présente pour les trois
        # entités, entièrement NaN pour l'Italie). Repris tel quel du notebook 3.
        df_country['climat_affaires'] = np.nan
        if params['climat_affaires_observe']:
            climat_noise = np.random.normal(0, 2.0, n_periods)
            df_country.loc[dates, 'climat_affaires'] = 100.0 + climat_noise

        df_country.loc[df_country.index[-1], 'inflation_ipc'] = np.nan
        df_country.loc[df_country.index[-1], 'taux_chomage'] = np.nan

        pib_available = df_country[df_country['pib_trimestriel'].notna()].index
        if len(pib_available) > 0:
            df_country.loc[pib_available[-1], 'pib_trimestriel'] = np.nan

        bc_available = df_country[df_country['balance_commerciale_annuelle'].notna()].index
        if len(bc_available) > 0:
            df_country.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan

        depenses_available = df_country[df_country['depenses_publiques_pib'].notna()].index
        if len(depenses_available) > 0:
            df_country.loc[depenses_available[-1], 'depenses_publiques_pib'] = np.nan

        all_data.append(df_country)

    df_panel = pd.concat(all_data, ignore_index=False)
    df_panel = df_panel.reset_index().rename(columns={'index': 'date'})
    df_panel = df_panel.set_index(['country', 'date'])
    df_panel = df_panel.sort_index()

    return df_panel

## 3 - Fonctions utilitaires d'affichage

`show(obj, titre)` affiche systématiquement le type, la forme, le nombre de NaN par colonne
et un `head`/`tail` de l'objet pandas passé en argument — c'est la fonction utilisée à
(presque) chaque étape ci-dessous pour inspecter les variables intermédiaires de `_fit` et
`_transform`. `section`/`subsection` ne font qu'imprimer des séparateurs visuels.

In [ ]:
def section(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def subsection(title):
    print("\n" + "-" * 90)
    print(title)
    print("-" * 90)


def show(obj, title=None, n=5, show_nan=True):
    if title:
        print(f"\n>>> {title}")
    if obj is None:
        print("  None")
        return
    if isinstance(obj, (pd.DataFrame, pd.Series)):
        print(f"  type={type(obj).__name__}  shape={obj.shape}")
        if show_nan:
            nan_count = obj.isna().sum()
            if isinstance(nan_count, pd.Series):
                print("  NaN par colonne :")
                display(nan_count.to_frame('n_nan'))
            else:
                print(f"  NaN : {nan_count} / {len(obj)}")
        print(f"  head({n}) :")
        display(obj.head(n))
        print(f"  tail({n}) :")
        display(obj.tail(n))
    else:
        print(f"  {obj}")

## 4 - Fonctions par phase

Chaque fonction ci-dessous correspond **exactement** à une section de `_fit` ou à
`_transform` dans `tsforecast/frequency/high_frequency_imputer.py` (mêmes noms de phase que
les commentaires `# PHASE 0`, ..., `# PHASE 6` du code source) : le corps de chaque fonction
appelle les méthodes privées réelles de l'imputer (`imputer._build_stage_frame(...)`,
`imputer._prepare_training_data(...)`, etc.) et n'ajoute que des instructions d'affichage.
Elles seront appelées une première fois sur les données de panel (section 5), puis une seconde
fois, à l'identique, sur les données de séries temporelles (section 8) — ce qui garantit que
les deux cheminements suivent rigoureusement le même algorithme.

In [ ]:
def phase0_setup(imputer, X, y=None):
    imputer.feature_columns_ = list(X.columns)
    imputer.target_column_ = y.name if y is not None else None
    imputer.is_panel_ = bool(imputer.panel_cols) or is_panel_data(data=X)
    print(f"is_panel_ = {imputer.is_panel_}")

    if y is not None:
        if len(X) != len(y):
            raise ValueError("X and y should be of equal length")
        X_work = pd.concat([X, y.to_frame()], axis=1)
    else:
        X_work = X.copy()

    if imputer.is_panel_ and isinstance(X.index, pd.MultiIndex):
        imputer.entities_ = get_unique_panel_entities(X)
    else:
        imputer.entities_ = None
    print(f"entities_ = {imputer.entities_}")

    try:
        index_freq = detect_index_frequency(X_work.index, return_format='base')
        imputer._source_index_frequency_label = imputer._stage_frequency_label(index_freq)
    except (ValueError, TypeError):
        imputer._source_index_frequency_label = None
    print(f"_source_index_frequency_label = {imputer._source_index_frequency_label}")

    if imputer.is_panel_ and isinstance(imputer.target_frequency, str) and imputer.entities_:
        imputer.effective_target_frequency_ = {entity: imputer.target_frequency for entity in imputer.entities_}
    elif isinstance(imputer.target_frequency, dict):
        imputer.effective_target_frequency_ = imputer.target_frequency.copy()
    else:
        imputer.effective_target_frequency_ = imputer.target_frequency
    print(f"effective_target_frequency_ (avant validation) = {imputer.effective_target_frequency_}")

    imputer.detected_frequencies_ = detect_frequency(data=X_work)
    if not imputer.detected_frequencies_:
        raise ValueError("Could not detect frequency for any column")
    subsection("detected_frequencies_")
    display(pd.Series(imputer.detected_frequencies_, name='frequence').to_frame())

    imputer.effective_target_frequency_ = imputer._target_freq_validator.validate(
        target_frequency=imputer.effective_target_frequency_,
        detected_frequencies=imputer.detected_frequencies_,
        on_frequency_mismatch=imputer.on_frequency_mismatch,
    )
    print(f"\neffective_target_frequency_ (apres validation) = {imputer.effective_target_frequency_}")

    imputer.variable_categories_ = imputer._classify_variables_at_frequency(
        imputer.effective_target_frequency_
    )
    subsection("variable_categories_")
    display(pd.Series(
        {key: category for category, keys in imputer.variable_categories_.items() for key in keys},
        name='categorie',
    ).to_frame())

    imputer.imputation_order_ = imputer._determine_imputation_order()
    subsection("imputation_order_")
    print(imputer.imputation_order_)

    return X_work

In [ ]:
def phase1_window(imputer, X_work):
    imputer._imputation_window_calc = ImputationWindowCalculator(
        coverage_threshold=imputer.coverage_threshold,
        imputation_scope=imputer.imputation_scope,
        min_columns=2,
    )
    try:
        imputer._imputation_window_calc.fit(X_work)
        imputer.imputation_window_ = imputer._zip_window_bounds(
            imputer._imputation_window_calc.imputation_strict_window_start_,
            imputer._imputation_window_calc.imputation_strict_window_end_,
        )
        imputer.training_window_ = imputer._zip_window_bounds(
            imputer._imputation_window_calc.imputation_window_start_,
            imputer._imputation_window_calc.imputation_window_end_,
        )
    except ValueError as e:
        warnings.warn(f"Could not calculate imputation window: {e}. Using all available data.", UserWarning)
        if isinstance(X_work.index, pd.MultiIndex):
            time_idx = X_work.index.get_level_values(-1)
        else:
            time_idx = X_work.index
        imputer.imputation_window_ = (time_idx.min(), time_idx.max())
        imputer.training_window_ = imputer.imputation_window_
    else:
        start = imputer._imputation_window_calc.imputation_strict_window_start_
        no_window = start is None if not isinstance(start, dict) else all(v is None for v in start.values())
        if no_window:
            warnings.warn(
                "No strict imputation window found: no model can be trained; "
                "all imputations will fall back to interpolation.", UserWarning
            )

    print("imputation_window_ (fenetre STRICTE, coverage==1.0) :")
    if isinstance(imputer.imputation_window_, dict):
        for entity, bounds in imputer.imputation_window_.items():
            print(f"  {entity} : {bounds}")
    else:
        print(f"  {imputer.imputation_window_}")

    print(f"\ntraining_window_ (fenetre ETENDUE, suit imputation_scope='{imputer.imputation_scope}') :")
    if isinstance(imputer.training_window_, dict):
        for entity, bounds in imputer.training_window_.items():
            print(f"  {entity} : {bounds}")
    else:
        print(f"  {imputer.training_window_}")

    mask = imputer._imputation_window_calc.get_imputation_window_mask(X_work)
    show(mask.to_frame('dans_la_fenetre'), "masque (True = dans la fenetre d'entrainement etendue)")

In [ ]:
def phase2_additive(imputer, X_work):
    if imputer.additive_transformer is not None:
        imputer.additive_transformer_ = clone(imputer.additive_transformer)
        X_work = imputer.additive_transformer_.fit_transform(X_work)
        if isinstance(X_work, tuple):
            X_work = X_work[0]
        show(X_work, "X_work apres additive_transformer_")
    else:
        imputer.additive_transformer_ = None
        print("additive_transformer=None : X_work n'est pas modifie a cette phase.")
    return X_work

In [ ]:
def phase3_freq_list(imputer):
    freq_prediction_list = imputer._build_frequency_prediction_list()
    print(f"Nombre d'etapes de la cascade : {len(freq_prediction_list)}")
    for i, pred_freq in enumerate(freq_prediction_list):
        print(f"  Etape {i} -> label='{imputer._stage_frequency_label(pred_freq)}' | pred_freq={pred_freq}")
    return freq_prediction_list

In [ ]:
def phase4_provenance_init(imputer, X_work):
    imputer._provenance_tracker = ImputationProvenanceTracker()
    imputer._provenance_tracker.initialize(X_work, panel_cols=imputer.panel_cols)
    prov_matrix_init = imputer._provenance_tracker.get_provenance_matrix()
    show(prov_matrix_init, "imputation_provenance (juste apres initialize, ORIGINAL vs None)")
    stats_init = imputer._provenance_tracker.compute_statistics()['overall']
    print("Statistiques globales initiales :")
    for prov_type in ProvenanceType:
        print(f"  {prov_type.value:>16s} : {stats_init[prov_type.value]:>5.0f}  ({stats_init[f'{prov_type.value}_pct']:.1f}%)")
    print(f"  {'not_imputed':>16s} : {stats_init['not_imputed']:>5.0f}  ({stats_init['not_imputed_pct']:.1f}%)")

In [ ]:
def phase5_cascade(imputer, X_work, freq_prediction_list, verbose_details=True):
    imputer.imputation_plan_ = []
    imputer.freq_prediction_list_ = freq_prediction_list
    imputed_store = {}

    for stage_num, pred_freq in enumerate(freq_prediction_list):
        stage_label = imputer._stage_frequency_label(pred_freq)
        section(f"ETAPE DE CASCADE {stage_num} - frequence '{stage_label}'")
        print(f"pred_freq brut = {pred_freq}")

        var_classification = imputer._classify_variables_at_frequency(pred_freq)
        aggregate_keys = var_classification['aggregate']
        impute_keys = var_classification['impute']
        print(f"\n5a. Classification a cette etape :")
        print(f"    a AGREGER   ({len(aggregate_keys)}) : {aggregate_keys}")
        print(f"    A IMPUTER   ({len(impute_keys)}) : {impute_keys}")
        print(f"    deja cible  ({len(var_classification['target_freq'])}) : {var_classification['target_freq']}")

        X_stage = imputer._build_stage_frame(X_work, imputed_store, pred_freq)
        show(X_stage, f"5b. X_stage reconstruit pour l'etape '{stage_label}'")
        imputer._mark_aggregated_provenance(imputer._provenance_tracker, X_stage, aggregate_keys)
        print("    -> provenance AGGREGATED marquee sur X_stage pour les colonnes agregees")

        if imputer.train_on_partial_fit_order == 'cv' and imputer.train_on_partial_coverage:
            ordered_impute_keys = imputer._determine_variable_order_cv(X_stage, impute_keys)
        else:
            ordered_impute_keys = sorted(
                impute_keys,
                key=lambda k: get_frequency_order(imputer.detected_frequencies_.get(k, 'D')),
                reverse=True,
            )
        print(f"\n5c. Ordre d'imputation au sein de l'etape : {ordered_impute_keys}")

        vars_in_stage = OrderedDict()
        for var_key in ordered_impute_keys:
            _, var_name = split_variable_key(var_key)
            freq_norm = normalize_frequency(imputer.detected_frequencies_[var_key], return_format='base')
            vars_in_stage.setdefault((var_name, freq_norm), []).append(var_key)

        freqs_per_var = {}
        for var_name, freq_norm in vars_in_stage:
            freqs_per_var.setdefault(var_name, set()).add(freq_norm)

        print(f"\n5d. Regroupement vars_in_stage (variable, freq_detectee) -> cles :")
        for (var_name, freq_norm), var_keys in vars_in_stage.items():
            print(f"    ({var_name!r}, {freq_norm!r}) -> {var_keys}")

        for (var_name, freq_norm), var_keys in vars_in_stage.items():
            group_key = var_name if len(freqs_per_var[var_name]) == 1 else (var_name, freq_norm)
            repr_var_key = var_keys[0]
            subsection(f"5d'. Groupe '{group_key}' (variable='{var_name}', freq_source='{freq_norm}', "
                        f"entites={var_keys})")

            stage_fields = dict(
                pred_freq_label=imputer._freq_label(pred_freq),
                pred_freq=pred_freq,
                var_key=group_key,
                var_name=var_name,
                source_frequency=freq_norm,
                entities=to_entity_tuple(
                    [split_variable_key(k)[0] for k in var_keys]
                    if imputer.is_panel_ and isinstance(var_keys[0], tuple) else None
                ),
            )
            stage_scale = imputer._stage_scale_factor(repr_var_key, pred_freq)
            print(f"    stage_fields = {stage_fields}")
            print(f"    stage_scale (nb de sous-periodes de l'etape par periode de la variable) = {stage_scale}")

            fallback_step = ImputationStep(
                **stage_fields, model=INTERPOLATE_FALLBACK, feature_cols=(), feature_means=None,
                scale_factor=stage_scale, fit_scale_factor=stage_scale, trained_on_imputed=False,
            )

            if not imputer.cascade_refitting:
                base = imputer._model_for_var(group_key)
                if base is not None:
                    print("    cascade_refitting=False et modele deja entraine -> reutilisation (replace scale_factor)")
                    imputer.imputation_plan_.append(replace(base, **stage_fields, scale_factor=stage_scale))
                    continue

            estimator = imputer._get_estimator_for_variable(var_name)
            if estimator is None:
                print("    Pas d'estimateur disponible -> repli interpolate_fallback")
                imputer.imputation_plan_.append(fallback_step)
                continue

            X_train, y_train, scale_factor, feature_factors = imputer._prepare_training_data(
                X_stage, X_work, repr_var_key, pred_freq
            )
            if verbose_details:
                show(X_train, "X_train (covariables agregees a f_var, restreintes a la fenetre d'entrainement)")
                show(y_train, "y_train (valeurs vraies de la variable, jamais ses propres imputations)")
                print(f"    scale_factor = {scale_factor}")
                print(f"    feature_factors (sous-periodes par covariable) :")
                display(feature_factors.to_frame('n_sous_periodes'))

            if len(X_train) < 2:
                print("    Jeu d'entrainement trop court (<2 obs) -> repli interpolate_fallback")
                imputer.imputation_plan_.append(fallback_step)
                continue

            X_train_scaled, y_train_scaled = imputer._apply_frequency_scaling(
                X_train, y_train, scale_factor, feature_factors
            )
            if verbose_details:
                show(X_train_scaled, "5e. X_train_scaled (apres mise a l'echelle des sous-periodes)")
                show(y_train_scaled, "5e. y_train_scaled")

            feature_means = X_train_scaled.mean()
            X_train_scaled = X_train_scaled.fillna(feature_means)
            X_train_scaled = X_train_scaled.dropna(axis=1, how='all')

            valid_mask = y_train_scaled.notna()
            X_train_scaled = X_train_scaled.loc[valid_mask]
            y_train_scaled = y_train_scaled.loc[valid_mask]
            if verbose_details:
                print(f"    feature_means (utilisees a la prediction, memorisees dans l'ImputationStep) :")
                display(feature_means.to_frame('moyenne'))
            print(f"    X_train_scaled final : shape={X_train_scaled.shape}, "
                  f"y_train_scaled final : shape={y_train_scaled.shape}")

            if len(X_train_scaled) < 2 or X_train_scaled.shape[1] == 0:
                print("    Donnees insuffisantes apres pretraitement -> repli interpolate_fallback")
                imputer.imputation_plan_.append(fallback_step)
                continue

            feature_cols = list(X_train_scaled.columns)
            try:
                estimator.fit(X_train_scaled, y_train_scaled)
                trained_on_imputed = imputer.train_on_partial_coverage and bool(imputed_store)
                step = ImputationStep(
                    **stage_fields, model=estimator, feature_cols=tuple(feature_cols),
                    feature_means=feature_means.reindex(feature_cols), scale_factor=scale_factor,
                    fit_scale_factor=scale_factor, trained_on_imputed=trained_on_imputed,
                )
                original_mask = imputer._provenance_tracker.get_mask(
                    [ProvenanceType.ORIGINAL, ProvenanceType.DISAGGREGATED], column=var_name
                ).reindex(y_train_scaled.index).fillna(False)
                n_true = int(original_mask.sum())
                print(f"    5f. Modele {type(estimator).__name__} entraine sur {len(y_train_scaled)} obs "
                      f"({n_true} vraies, {len(y_train_scaled) - n_true} imputees), "
                      f"trained_on_imputed={trained_on_imputed}")
                if hasattr(estimator, 'coef_'):
                    print(f"        coef_={np.round(estimator.coef_, 4)}  intercept_={estimator.intercept_:.4f}")
            except Exception as e:
                print(f"    Echec du fit ({e!r}) -> repli interpolate_fallback")
                step = fallback_step

            imputer.imputation_plan_.append(step)
            print(f"    -> ImputationStep enregistre dans imputation_plan_ "
                  f"(taille actuelle = {len(imputer.imputation_plan_)})")

            if imputer.cascade_refitting and not step.is_fallback:
                stage_group = step.group_metadata()
                stage_context = f"'{var_name}' at stage {stage_label}"
                scope_mask, predict_mask = imputer._prediction_masks(
                    X_stage, X_work, stage_group, var_name,
                    imputer._imputation_window_calc, context=stage_context,
                )
                print(f"\n    5g. Cascade refitting : scope_mask.sum()={scope_mask.sum()}, "
                      f"predict_mask.sum()={predict_mask.sum()}")
                if scope_mask.any():
                    try:
                        preds = imputer._predict_stage_values(step, X_stage, predict_mask, context=stage_context)
                        if verbose_details:
                            show(preds, "5g. preds (predictions brutes du modele, avant contrainte additive)")

                        preds, disagg_mask = imputer._apply_period_totals(preds, X_work, stage_group, context=stage_context)
                        if verbose_details:
                            show(preds, "5g. preds (apres contrainte additive enforce_period_totals)")
                        print(f"        cellules recalees (DISAGGREGATED) = {int(disagg_mask.sum())} / {len(disagg_mask)}")

                        # Vidage (scope INTER fenetre) puis reecriture et marquage :
                        # meme point de passage que la classe. Le marquage porte sur
                        # l'UNION des cellules recalees et des dates-ancres du
                        # perimetre - une ancre reste une ancre, que le recalage ait
                        # eu lieu ou non (§3.15, B2)
                        anchor_mask = imputer._anchor_mask(X_work, var_name, preds.index)
                        written = imputer._write_stage_values(
                            X_stage, X_work, var_name, preds, scope_mask, predict_mask,
                            imputer._provenance_tracker, disagg_mask,
                            step.trained_on_imputed, context=stage_context,
                        )
                        marked = disagg_mask | anchor_mask
                        print(f"        -> provenance marquee : {int(marked.sum())} DISAGGREGATED "
                              f"(dont {int(anchor_mask.sum())} dates-ancres), "
                              f"{int((~marked).sum())} MODEL_ON_*")

                        # Le RECEVEUR du combine_first l'emporte : la prediction de
                        # l'etape COURANTE gagne, a l'echelle la plus fine atteinte.
                        # "existing" ne subsiste que sur les lignes d'un AUTRE groupe
                        # de la meme variable (§3.15, B5)
                        if written:
                            existing_imputed = imputed_store.get(var_name)
                            imputed_store[var_name] = (
                                preds if existing_imputed is None else preds.combine_first(existing_imputed)
                            )
                            if verbose_details:
                                show(imputed_store[var_name], f"5g. imputed_store['{var_name}'] mis a jour "
                                                                f"(alimente les etapes suivantes de la cascade)")
                    except Exception as e:
                        warnings.warn(f"Intermediate imputation failed for '{var_name}': {e}")

    return imputed_store

In [ ]:
def phase6_finalize(imputer):
    imputer.imputation_provenance_fit_ = imputer._provenance_tracker.get_provenance_matrix()
    show(imputer.imputation_provenance_fit_, "imputation_provenance_fit_ (provenance a la fin du fit)")

    print(f"\nimputation_plan_ final : {len(imputer.imputation_plan_)} etapes enregistrees")
    plan_summary = pd.DataFrame([
        {
            'stage_freq': step.pred_freq_label if isinstance(step.pred_freq_label, str)
                          else imputer._stage_frequency_label(step.pred_freq),
            'var_key': step.var_key,
            'var_name': step.var_name,
            'source_frequency': step.source_frequency,
            'is_fallback': step.is_fallback,
            'scale_factor': step.scale_factor,
            'fit_scale_factor': step.fit_scale_factor,
            'trained_on_imputed': step.trained_on_imputed,
            'n_features': len(step.feature_cols),
        }
        for step in imputer.imputation_plan_
    ])
    display(plan_summary)
    return plan_summary

In [ ]:
def run_transform(imputer, X, y=None, verbose_details=True):
    imputer._original_X_ = X.copy()
    imputer._original_y_ = y.copy() if y is not None else None

    y_col_name = None
    if y is not None:
        y_col_name = imputer._resolve_target_column_name(y)
        y = imputer._align_target_index(X, y)
        data_work = pd.concat([X, y.to_frame(name=y_col_name)], axis=1)
    else:
        data_work = X.copy()

    if not isinstance(data_work.index, (pd.DatetimeIndex, pd.MultiIndex)):
        if imputer.time_col and imputer.time_col in data_work.columns:
            data_work = data_work.set_index(imputer.time_col)
        else:
            raise ValueError("Data must have a DatetimeIndex or MultiIndex")

    # Fenetre d'imputation calculee AVANT le transformateur additif, au meme
    # stade que la PHASE 1 du fit
    transform_window_calc, window_error = imputer._fit_imputation_window(data_work)
    if transform_window_calc is None:
        warnings.warn(f"Could not calculate the imputation window: {window_error}.")

    if imputer.additive_transformer_ is not None:
        data_transformed = imputer.additive_transformer_.transform(data_work)
        if isinstance(data_transformed, tuple):
            data_transformed = data_transformed[0]
    else:
        data_transformed = data_work.copy()
        print("additive_transformer_ est None : data_transformed = data_work (copie)")

    # Tracker initialise APRES le transformateur additif, comme la PHASE 4 du
    # fit suit sa PHASE 2 : ORIGINAL marque les valeurs presentes a l'entree de
    # la CASCADE, et la matrice partage l'index et les colonnes des frames
    # d'etape, tous derives de data_transformed (§3.15, B8)
    transform_tracker = ImputationProvenanceTracker()
    transform_tracker.initialize(data_transformed, panel_cols=imputer.panel_cols)
    show(transform_tracker.get_provenance_matrix(), "provenance initiale du transform (ORIGINAL vs None)")

    section("_transform - Rejeu des etapes de la cascade (freq_prediction_list_)")

    X_input = data_transformed
    imputed_store_t = {}
    stage_frames = OrderedDict()
    provenance_frames = OrderedDict()

    for stage_idx, pred_freq in enumerate(imputer.freq_prediction_list_):
        var_classification = imputer._classify_variables_at_frequency(pred_freq)
        aggregate_keys = var_classification['aggregate']

        X_stage = imputer._build_stage_frame(X_input, imputed_store_t, pred_freq)
        imputer._mark_aggregated_provenance(transform_tracker, X_stage, aggregate_keys)

        # Frame des COVARIABLES, miroir exact du X_stage du fit : X_stage porte
        # la sortie et recoit toutes les ecritures, le miroir n'accumule que ce
        # que le bloc 5g accumule - etapes NON de repli, et seulement sous
        # cascade_refitting (§3.15, B7)
        X_covariates = X_stage.copy()

        registry_label = imputer._freq_label(pred_freq)
        stage_steps = [step for step in imputer.imputation_plan_ if step.pred_freq_label == registry_label]

        freq_label = imputer._stage_frequency_label(pred_freq)
        is_final_stage = stage_idx == len(imputer.freq_prediction_list_) - 1

        section(f"ETAPE DE REPLAY {stage_idx} - frequence '{freq_label}' "
                f"({'DERNIERE ETAPE = frequence cible' if is_final_stage else 'etape intermediaire'})")
        show(X_stage, f"X_stage reconstruit pour l'etape '{freq_label}'")
        print(f"Variables du plan rejouees a cette etape : {[s.var_name for s in stage_steps]}")

        for step in stage_steps:
            var_name = step.var_name
            stage_group = step.group_metadata()
            stage_context = f"'{var_name}' at stage {freq_label}"
            subsection(f"Rejeu de '{var_name}' (var_key={step.var_key}, is_fallback={step.is_fallback})")

            if step.is_fallback:
                print("    Etape de repli (interpolate_fallback) : interpolation lineaire + recalage additif")
                if var_name in X_stage.columns:
                    # Interpolation lue sur le miroir : le fit ne produit aucune
                    # valeur pour un repli, les etapes suivantes ne doivent donc
                    # pas la voir. Seule la sortie la recoit
                    interpolated = X_covariates[var_name].interpolate(method='linear', limit_direction='both')
                    scope_mask, predict_mask = imputer._prediction_masks(
                        X_stage, X_input, stage_group, var_name,
                        transform_window_calc, context=stage_context,
                    )
                    rescaled, disagg_mask = imputer._apply_period_totals(
                        interpolated.loc[predict_mask], X_input, stage_group, context=stage_context
                    )
                    imputer._write_stage_values(
                        X_stage, X_input, var_name, rescaled, scope_mask, predict_mask,
                        transform_tracker, disagg_mask, False, context=stage_context,
                    )
                    if verbose_details:
                        show(rescaled, "rescaled (valeurs interpolees puis recalees)")
                continue

            if var_name not in X_stage.columns:
                print(f"    '{var_name}' absent de X_stage -> rien a faire")
                continue

            scope_mask, predict_mask = imputer._prediction_masks(
                X_stage, X_input, stage_group, var_name,
                transform_window_calc, context=stage_context,
            )
            print(f"    scope_mask.sum()={scope_mask.sum()}  predict_mask.sum()={predict_mask.sum()}")
            if not scope_mask.any():
                print("    scope_mask vide -> rien a faire")
                continue

            try:
                # Covariables lues sur le miroir, dont l'etat reproduit celui du
                # frame d'etape du fit au meme instant de la cascade
                predictions = imputer._predict_stage_values(step, X_covariates, predict_mask, context=stage_context)
                if verbose_details:
                    show(predictions, "predictions brutes du modele")

                predictions, disagg_mask = imputer._apply_period_totals(
                    predictions, X_input, stage_group, context=stage_context
                )
                if verbose_details:
                    show(predictions, "predictions apres contrainte additive (enforce_period_totals)")
                print(f"    cellules DISAGGREGATED = {int(disagg_mask.sum())} / {len(disagg_mask)}")

                # Le miroir n'est mis a jour que sous la meme garde que le bloc
                # 5g du fit : etape non de repli ET cascade_refitting
                mirror = (X_covariates,) if imputer.cascade_refitting and not step.is_fallback else ()
                written = imputer._write_stage_values(
                    X_stage, X_input, var_name, predictions, scope_mask, predict_mask,
                    transform_tracker, disagg_mask, step.trained_on_imputed,
                    extra_frames=mirror, context=stage_context,
                )

                # Sens du combine_first : la prediction de l'etape COURANTE gagne
                if imputer.cascade_refitting and written:
                    existing_imputed = imputed_store_t.get(var_name)
                    imputed_store_t[var_name] = (
                        predictions if existing_imputed is None else predictions.combine_first(existing_imputed)
                    )
                    if verbose_details:
                        show(imputed_store_t[var_name], f"imputed_store['{var_name}'] mis a jour")
            except Exception as e:
                warnings.warn(f"Prediction failed for variable '{var_name}': {e}. Using interpolation fallback.")
                # Meme discipline que le repli declare : lecture sur le miroir,
                # ecriture restreinte aux lignes predictibles et tracee
                interpolated = X_covariates[var_name].interpolate(method='linear', limit_direction='both')
                filled = interpolated.loc[predict_mask].dropna()
                imputer._write_stage_values(
                    X_stage, X_input, var_name, filled, scope_mask, predict_mask,
                    transform_tracker, pd.Series(False, index=filled.index),
                    step.trained_on_imputed, context=stage_context,
                )

        if stage_steps or is_final_stage:
            stage_frames[freq_label] = X_stage.copy()
            provenance_frames[freq_label] = transform_tracker.get_provenance_matrix()
            print(f"\n-> frame de l'etape '{freq_label}' enregistre dans stage_frames "
                  f"(niveaux stockes jusqu'ici : {list(stage_frames.keys())})")

        data_transformed = X_stage
        final_stage_label = freq_label

    section("_transform - Construction de la sortie finale")

    if imputer.keep_lower_frequencies and stage_frames:
        data_result = imputer._build_multifreq_output(stage_frames)
        print("keep_lower_frequencies=True -> sortie MultiIndex empilant tous les niveaux de frequence")
    else:
        data_result = stage_frames[final_stage_label]
        print("keep_lower_frequencies=False -> sortie a la seule frequence cible")

    show(data_result, "data_result (sortie finale de _transform)")

    if imputer.keep_lower_frequencies and provenance_frames:
        imputer.imputation_provenance_ = imputer._build_multifreq_output(provenance_frames)
    else:
        imputer.imputation_provenance_ = transform_tracker.get_provenance_matrix()
    show(imputer.imputation_provenance_, "imputation_provenance_ (meme structure d'index que data_result)")

    overall_stats = transform_tracker.compute_statistics()['overall']
    print("\nResume de provenance final (transform) :")
    for prov_type in ProvenanceType:
        print(f"  {prov_type.value:>16s} : {overall_stats[prov_type.value]:>6.0f}  ({overall_stats[f'{prov_type.value}_pct']:.1f}%)")
    print(f"  {'not_imputed':>16s} : {overall_stats['not_imputed']:>6.0f}  ({overall_stats['not_imputed_pct']:.1f}%)")

    if y is not None and y_col_name in data_result.columns:
        y_transformed = data_result[y_col_name]
        X_transformed = data_result.drop(columns=[y_col_name])
        return data_result, X_transformed, y_transformed
    else:
        return data_result, data_result, None

---
## 5 - Partie panel : `_fit` pas à pas

C'est le cas le plus complexe : 3 pays (France, Allemagne, Italie), chacun avec sa propre
période de couverture, et `depenses_publiques_pib` publiée **annuellement** pour la France et
l'Italie mais **trimestriellement** pour l'Allemagne — ce qui va déclencher, à l'étape
mensuelle, le cas où une même variable se scinde en deux groupes d'entraînement distincts
(`(depenses_publiques_pib, Y)` et `(depenses_publiques_pib, Q)`), documenté au §2.4 de la
docstring de la classe.

On configure l'imputer avec `cascade_refitting=True` et `keep_lower_frequencies=True` pour
observer la cascade au complet (les valeurs imputées à une étape basse fréquence alimentent
les étapes suivantes, et la sortie finale empile tous les niveaux de fréquence intermédiaires).

In [ ]:
df_panel = create_panel_dataset()

# Le pas a pas ci-dessous deroule l'ANCIEN HighFrequencyImputer, qui ne sait pas
# traiter une covariable structurellement absente pour toute une entite
# (`climat_affaires` n'est jamais observee pour l'Italie). C'est precisement la
# limite que leve HighFrequencyImputer2 via `covariate_eligibility`
# (high_frequency_imputer2_architecture.md, §4.5) ; elle sera illustree dans le
# futur notebook 5. On ecarte donc la colonne ici, sans toucher au generateur
# (qui reste synchronise avec le notebook 3).
df_panel = df_panel.drop(columns=['climat_affaires'])

show(df_panel, "df_panel (donnees brutes)")

In [ ]:
imputer_panel = HighFrequencyImputer(
    target_frequency='M',
    estimator=LinearRegression(),
    cascade_refitting=True,
    keep_lower_frequencies=True,
    imputation_scope='extended_forward',
    coverage_threshold=0.5,
    train_on_partial_coverage=False,
    scale_features=True,
    enforce_period_totals=True,
    verbose=True,
)
imputer_panel

### PHASE 0 — Setup

Reprend exactement le début de `_fit` : détection panel/non-panel, extraction des entités,
détection de la fréquence de chaque colonne (`detected_frequencies_`), expansion de
`target_frequency` en dictionnaire par entité (`effective_target_frequency_`), puis
classification de chaque variable en `aggregate` / `impute` / `target_freq`
(`variable_categories_`) et détermination de l'ordre d'imputation (`imputation_order_`,
des fréquences les plus basses aux plus hautes).

In [ ]:
X_work_panel = phase0_setup(imputer_panel, df_panel)

### PHASE 1 — Fenêtre d'imputation

`ImputationWindowCalculator` calcule deux fenêtres :
- la fenêtre **stricte** (`imputation_window_`) où *toutes* les colonnes ont une vraie
  observation (`coverage == 1.0`) — utilisée pour délimiter ce qui peut servir de vérité
  terrain ;
- la fenêtre **étendue** (`training_window_`), qui suit `imputation_scope` (ici
  `'extended_forward'` avec `coverage_threshold=0.5`) — elle va au-delà de la fenêtre stricte
  pour couvrir les fins de série retardées.

Ces deux fenêtres diffèrent par entité, la couverture temporelle du panel étant hétérogène.

In [ ]:
phase1_window(imputer_panel, X_work_panel)

### PHASE 2 — Transformateur additif

Ici `additive_transformer=None` : cette phase est un no-op, `X_work` n'est pas modifié. Elle
sert normalement à rendre les données additives (log, différenciation, ...) avant la cascade.

In [ ]:
X_work_panel = phase2_additive(imputer_panel, X_work_panel)

### PHASE 3 — Liste des fréquences de la cascade (`freq_prediction_list_`)

Construit la liste ordonnée des fréquences auxquelles une prédiction doit être faite pour
atteindre la fréquence cible, de la plus basse à la cible. Pour un panel, chaque étape est un
dictionnaire `entité -> fréquence` : selon `effective_target_frequency_`, une entité peut ne
pas être concernée par une étape intermédiaire donnée.

In [ ]:
freq_prediction_list_panel = phase3_freq_list(imputer_panel)

### PHASE 4 — Initialisation du suivi de provenance

`ImputationProvenanceTracker` crée une matrice parallèle aux données : chaque cellule non-NaN
de `X_work` est marquée `ORIGINAL`, chaque cellule NaN reste `None` (à remplir au fur et à
mesure de la cascade). C'est cette matrice qui, en fin de `_fit`, devient
`imputation_provenance_fit_`.

In [ ]:
phase4_provenance_init(imputer_panel, X_work_panel)

### PHASE 5 — Boucle de cascade (cœur de l'algorithme)

Pour chaque fréquence de l'étape (`freq_prediction_list_`), dans l'ordre :

- **5a.** classification des variables à cette fréquence précise (`aggregate` / `impute`) ;
- **5b.** reconstruction de `X_stage` **depuis les données d'origine** (jamais depuis l'étape
  précédente, pour ne pas accumuler d'artefacts d'agrégation), avec injection des valeurs déjà
  imputées aux étapes précédentes (`imputed_store`) ;
- **5c.** ordre d'imputation des variables au sein de l'étape ;
- **5d.** regroupement des clés par `(variable, fréquence détectée)` — le modèle est **global
  au panel**, jamais entraîné entité par entité ; c'est ce regroupement qui scinde
  `depenses_publiques_pib` en deux groupes distincts à l'étape mensuelle (Allemagne en `Q`,
  France/Italie en `Y`) ;
- **5e.** préparation et mise à l'échelle des données d'entraînement (`X_train`, `y_train`,
  `scale_factor`) ;
- **5f.** entraînement du modèle (ou repli `interpolate_fallback` si les données manquent) et
  enregistrement d'un `ImputationStep` dans `imputation_plan_` ;
- **5g.** si `cascade_refitting=True`, prédiction immédiate sur le périmètre imputable, mise
  sous contrainte additive (`enforce_period_totals`) et alimentation d'`imputed_store` pour les
  étapes suivantes.

**Attention : cette cellule produit une trace complète et volontairement verbeuse** — c'est
elle qui répond à la demande de voir "à chaque étape... la valeur des différentes variables".

In [ ]:
imputed_store_panel = phase5_cascade(imputer_panel, X_work_panel, freq_prediction_list_panel, verbose_details=True)

### PHASE 6 — Finalisation

`imputation_provenance_fit_` fige la matrice de provenance telle qu'observée à la fin du fit.
`imputation_plan_` est la seule source de vérité de l'état entraîné : `imputation_models_`,
`model_fitting_order_`, `stage_groups_` et `frequency_progression_` en sont des vues dérivées.
Le tableau récapitulatif ci-dessous liste chaque étape enregistrée (une ligne par couple
étape/groupe de variable).

In [ ]:
plan_summary_panel = phase6_finalize(imputer_panel)

## 6 - Partie panel : `_transform` pas à pas

`_transform` **rejoue** exactement les étapes enregistrées dans `imputation_plan_` et
`freq_prediction_list_` — jamais recalculées. Pour chaque étape : reconstruction de `X_stage`
(même méthode `_build_stage_frame` qu'au fit), puis pour chaque variable du plan, soit
interpolation linéaire (étape de repli), soit prédiction du modèle suivie du recalage additif
(`_apply_period_totals`). `keep_lower_frequencies=True` empile ensuite tous les niveaux de
fréquence dans un unique DataFrame à MultiIndex `(entité, frequency, date)`.

On transforme ici le même jeu de données que celui utilisé pour le fit — un usage classique de
`fit_transform`.

In [ ]:
data_result_panel, X_transformed_panel, y_transformed_panel = run_transform(
    imputer_panel, df_panel, y=None, verbose_details=True
)

### Aperçu de la sortie finale par niveau de fréquence (entité France)

`data_result_panel` empile tous les niveaux de fréquence de la cascade. On isole ici, pour la
France, chaque niveau (`Q` puis `M`) afin de voir concrètement comment la ligne trimestrielle
d'origine se retrouve désagrégée en observations mensuelles dont la somme reconstitue le total
trimestriel observé (colonnes `pib_trimestriel` / `depenses_publiques_pib` notamment).

In [ ]:
for freq_lvl in data_result_panel.index.get_level_values('frequency').unique():
    # Les niveaux d'entite gardent le nom des colonnes d'origine ('country'),
    # ils ne sont jamais renommes en 'entity' par l'empilement multi-frequences
    entity_level = data_result_panel.index.names[0]
    sub = data_result_panel.xs(('France', freq_lvl), level=(entity_level, 'frequency'))
    show(sub, f"niveau frequency='{freq_lvl}' pour France", n=4)

## 7 - Vérification croisée (panel) : pas à pas vs `fit_transform()`

Le déroulé manuel ci-dessus appelle les **mêmes méthodes privées** que `_fit`/`_transform`,
mais rien ne garantit qu'aucune étape n'a été oubliée. On le vérifie directement : un second
imputer, identique, entraîné et transformé via l'API publique standard
(`imputer.fit_transform(df_panel)`), doit produire un résultat **rigoureusement identique**
(même index, mêmes colonnes, écart absolu maximal nul).

In [ ]:
imputer_panel_ref = HighFrequencyImputer(
    target_frequency='M',
    estimator=LinearRegression(),
    cascade_refitting=True,
    keep_lower_frequencies=True,
    imputation_scope='extended_forward',
    coverage_threshold=0.5,
    train_on_partial_coverage=False,
    scale_features=True,
    enforce_period_totals=True,
    verbose=False,
)
ref_result_panel = imputer_panel_ref.fit_transform(df_panel)

diff_panel = (
    data_result_panel.reindex(ref_result_panel.index)[ref_result_panel.columns] - ref_result_panel
).abs()

print(f"shape pas-a-pas       : {data_result_panel.shape}")
print(f"shape fit_transform() : {ref_result_panel.shape}")
print(f"index identique       : {data_result_panel.index.equals(ref_result_panel.index)}")
print(f"colonnes identiques   : {list(data_result_panel.columns) == list(ref_result_panel.columns)}")
print(f"ecart absolu maximal  : {diff_panel.max().max()}")
print(f"nb etapes du plan (pas-a-pas vs fit_transform) : "
      f"{len(imputer_panel.imputation_plan_)} vs {len(imputer_panel_ref.imputation_plan_)}")

assert data_result_panel.index.equals(ref_result_panel.index)
assert diff_panel.max().max() == 0.0
print("\nOK : le deroule pas-a-pas reproduit exactement fit_transform().")

# §6 : le replay doit aussi etre stable entre `fit_transform(X)` et
# `fit(X)` suivi de `transform(X)` sur une instance separee. C'est ce que
# garantit la symetrie des gardes de cascade entre `_fit` et `_transform`
# (§3.15, B7) : sans elle, les covariables vues a l'entrainement et celles
# vues au replay divergent des que `cascade_refitting=False`
imputer_panel_split = clone(imputer_panel_ref)
imputer_panel_split.fit(df_panel)
split_result_panel = imputer_panel_split.transform(df_panel)

diff_split_panel = (ref_result_panel - split_result_panel.reindex(ref_result_panel.index)).abs()
print("")
print(f"fit_transform() vs fit()+transform() : ecart absolu maximal = "
      f"{diff_split_panel.max().max()}")
assert ref_result_panel.index.equals(split_result_panel.index)
assert diff_split_panel.max().max() == 0.0
print("OK : fit_transform() et fit()+transform() coincident cellule a cellule.")


---
## 8 - Partie séries temporelles : `_fit` et `_transform` pas à pas

Même jeu de fonctions par phase (section 4), appliqué cette fois au jeu de données de séries
temporelles simples (une seule "entité" implicite, pas de MultiIndex) : PIB trimestriel,
inflation et chômage mensuels, production industrielle mensuelle avec historique tronqué, et
balance commerciale annuelle avec un historique antérieur au reste (index irrégulier). C'est
un cas plus simple que le panel (pas de désaccord de fréquence entre entités, pas de
`(variable, fréquence)` en clé de groupe), mais qui illustre la même mécanique de cascade sur
un jeu de données à une seule série.

In [ ]:
df_ts = create_timeseries_dataset()
show(df_ts, "df_ts (donnees brutes)")

In [ ]:
imputer_ts = HighFrequencyImputer(
    target_frequency='M',
    estimator=LinearRegression(),
    cascade_refitting=True,
    keep_lower_frequencies=True,
    imputation_scope='extended_forward',
    coverage_threshold=0.5,
    train_on_partial_coverage=False,
    scale_features=True,
    enforce_period_totals=True,
    verbose=True,
)
imputer_ts

### PHASE 0 — Setup

In [ ]:
X_work_ts = phase0_setup(imputer_ts, df_ts)

### PHASE 1 — Fenêtre d'imputation

Série unique : les bornes de fenêtre sont ici des scalaires, pas des dictionnaires par entité.

In [ ]:
phase1_window(imputer_ts, X_work_ts)

### PHASE 2 — Transformateur additif (no-op ici aussi)

In [ ]:
X_work_ts = phase2_additive(imputer_ts, X_work_ts)

### PHASE 3 — Liste des fréquences de la cascade

In [ ]:
freq_prediction_list_ts = phase3_freq_list(imputer_ts)

### PHASE 4 — Initialisation du suivi de provenance

In [ ]:
phase4_provenance_init(imputer_ts, X_work_ts)

### PHASE 5 — Boucle de cascade

Comme pour le panel : reconstruction de `X_stage` à chaque étape, entraînement (ou repli),
puis, avec `cascade_refitting=True`, prédiction intermédiaire et mise à jour d'`imputed_store`.

In [ ]:
imputed_store_ts = phase5_cascade(imputer_ts, X_work_ts, freq_prediction_list_ts, verbose_details=True)

### PHASE 6 — Finalisation

In [ ]:
plan_summary_ts = phase6_finalize(imputer_ts)

## 9 - Partie séries temporelles : `_transform` pas à pas

In [ ]:
data_result_ts, X_transformed_ts, y_transformed_ts = run_transform(
    imputer_ts, df_ts, y=None, verbose_details=True
)

### Aperçu de la sortie finale par niveau de fréquence

In [ ]:
for freq_lvl in data_result_ts.index.get_level_values('frequency').unique():
    sub = data_result_ts.xs(freq_lvl, level='frequency')
    show(sub, f"niveau frequency='{freq_lvl}'", n=4)

## 10 - Vérification croisée (séries temporelles) : pas à pas vs `fit_transform()`

In [ ]:
imputer_ts_ref = HighFrequencyImputer(
    target_frequency='M',
    estimator=LinearRegression(),
    cascade_refitting=True,
    keep_lower_frequencies=True,
    imputation_scope='extended_forward',
    coverage_threshold=0.5,
    train_on_partial_coverage=False,
    scale_features=True,
    enforce_period_totals=True,
    verbose=False,
)
ref_result_ts = imputer_ts_ref.fit_transform(df_ts)

diff_ts = (
    data_result_ts.reindex(ref_result_ts.index)[ref_result_ts.columns] - ref_result_ts
).abs()

print(f"shape pas-a-pas       : {data_result_ts.shape}")
print(f"shape fit_transform() : {ref_result_ts.shape}")
print(f"index identique       : {data_result_ts.index.equals(ref_result_ts.index)}")
print(f"colonnes identiques   : {list(data_result_ts.columns) == list(ref_result_ts.columns)}")
print(f"ecart absolu maximal  : {diff_ts.max().max()}")
print(f"nb etapes du plan (pas-a-pas vs fit_transform) : "
      f"{len(imputer_ts.imputation_plan_)} vs {len(imputer_ts_ref.imputation_plan_)}")

assert data_result_ts.index.equals(ref_result_ts.index)
assert diff_ts.max().max() == 0.0
print("\nOK : le deroule pas-a-pas reproduit exactement fit_transform().")

# §6 : le replay doit aussi etre stable entre `fit_transform(X)` et
# `fit(X)` suivi de `transform(X)` sur une instance separee. C'est ce que
# garantit la symetrie des gardes de cascade entre `_fit` et `_transform`
# (§3.15, B7) : sans elle, les covariables vues a l'entrainement et celles
# vues au replay divergent des que `cascade_refitting=False`
imputer_ts_split = clone(imputer_ts_ref)
imputer_ts_split.fit(df_ts)
split_result_ts = imputer_ts_split.transform(df_ts)

diff_split_ts = (ref_result_ts - split_result_ts.reindex(ref_result_ts.index)).abs()
print("")
print(f"fit_transform() vs fit()+transform() : ecart absolu maximal = "
      f"{diff_split_ts.max().max()}")
assert ref_result_ts.index.equals(split_result_ts.index)
assert diff_split_ts.max().max() == 0.0
print("OK : fit_transform() et fit()+transform() coincident cellule a cellule.")


---
## 11 - Pour aller plus loin

Quelques leviers à modifier dans les cellules de configuration (`imputer_panel = ...` /
`imputer_ts = ...`) ci-dessus pour explorer d'autres comportements, en ré-exécutant ensuite les
phases 0 à 6 et `_transform` :

- **`cascade_refitting=False`** : chaque variable n'est entraînée qu'une seule fois (à la
  première étape où elle est imputable) puis réutilisée aux étapes suivantes — observer, dans
  le tableau `imputation_plan_`, que `fit_scale_factor` reste alors constant pour une variable
  donnée alors que `scale_factor` change d'une étape à l'autre. Le drapeau pilote aussi la
  **cascade intra-étape**, à l'identique au `fit` et au `transform` : les variables imputées
  plus tard dans une étape ne voient les valeurs produites par les précédentes que sous
  `cascade_refitting=True`. Une étape de **repli** ne les alimente jamais, quel que soit le
  drapeau : c'est le rôle du frame `X_covariates`, miroir du `X_stage` du fit (§3.15, B7).
- **`train_on_partial_coverage=True`** : les modèles peuvent s'entraîner sur des valeurs déjà
  imputées à une étape précédente hors de la fenêtre stricte — observer `trained_on_imputed`
  passer à `True` dans le plan, et la provenance basculer de `model_on_true` à
  `model_on_mixed`.
- **`enforce_period_totals=False`** : désactive le recalage additif (PHASE 5g / réplication en
  `_transform`) — observer que les sous-périodes ne somment plus à la valeur observée de la
  période basse fréquence, mais que la colonne reste **homogène en échelle** : l'option ne
  pilote que le recalage, jamais le marquage. Les **dates-ancres** restent `disaggregated`
  (elles portent là où une observation réelle a été lue) et seules les sous-périodes basculent
  en `model_on_true`/`model_on_mixed`. Sans cela, le filtre de provenance de
  `_prepare_training_data` vidait le masque d'entraînement de l'étape suivante et envoyait
  toute la variable en repli interpolation (§3.15, B2).
- **`keep_lower_frequencies=False`** : la sortie de `_transform` redevient un simple DataFrame
  à la fréquence cible, sans MultiIndex `(entité, frequency, date)`.
- **`imputation_scope`** (`'strict'`, `'extended_backward'`, `'extended_forward'`,
  `'extended_both'`) et **`coverage_threshold`** : modifient directement `training_window_`
  calculé en PHASE 1, donc l'étendue des données réellement disponibles pour l'entraînement.